# Preprocessing danych w scikit-learn

**Plan notatnika:**
1. Po co preprocessing
2. Import bibliotek i dane
3. Zmienna docelowa – `LabelEncoder`
4. Cechy porządkowe – `OrdinalEncoder`
5. Cechy nominalne – `OneHotEncoder`
6. `pd.get_dummies()` a `OneHotEncoder`
7. Skalowanie – `StandardScaler` i `MinMaxScaler`
8. Zasada: `fit` tylko na zbiorze treningowym
9. Wszystko razem – `ColumnTransformer`
10. Podsumowanie

Dokumentacja: [scikit-learn.org/stable/modules/preprocessing.html](https://scikit-learn.org/stable/modules/preprocessing.html)

## 1. Po co preprocessing

Modele scikit-learn pracują na **liczbach**. Prawdziwe dane zawierają jednak tekst (`'red'`, `'XL'`) oraz kolumny o bardzo różnych skalach (cena rzędu 100, waga rzędu 400). Przed treningiem trzeba je przygotować:

| Problem | Rozwiązanie |
|---|---|
| Kategorie tekstowe | kodowanie: `LabelEncoder`, `OrdinalEncoder`, `OneHotEncoder` |
| Różne skale cech | skalowanie: `StandardScaler`, `MinMaxScaler` |
| Różne kolumny wymagają różnych operacji | `ColumnTransformer` |

**Wspólny schemat wszystkich transformerów scikit-learn:**
- `fit(X)` – uczy się parametrów z danych (np. średniej i odchylenia, listy kategorii),
- `transform(X)` – stosuje wyuczone parametry,
- `fit_transform(X)` – oba kroki naraz.

Wyuczone parametry są dostępne w atrybutach zakończonych znakiem `_`, np. `scaler.mean_`.

## 2. Import bibliotek i dane

In [1]:
import numpy as np
import pandas as pd
import sklearn

sklearn.__version__

'1.9.0'

In [2]:
data = {
    'size': ['XL', 'L', 'M', 'L', 'M'],
    'color': ['red', 'green', 'blue', 'green', 'red'],
    'gender': ['female', 'male', 'male', 'female', 'female'],
    'price': [199.0, 89.0, 99.0, 129.0, 79.0],
    'weight': [500, 450, 300, 380, 410],
    'bought': ['yes', 'no', 'yes', 'no', 'yes']
}

df_raw = pd.DataFrame(data)
df = df_raw.copy()   # pracujemy na kopii, oryginał zostaje do porównań
df

,size,color,gender,price,weight,bought
0,XL,red,female,199.0,500,yes
1,L,green,male,89.0,450,no
2,M,blue,male,99.0,300,yes
3,L,green,female,129.0,380,no
4,M,red,female,79.0,410,yes


Rodzaje kolumn w tym zbiorze:
- `size` – kategoria **porządkowa** (M < L < XL),
- `color`, `gender` – kategorie **nominalne** (bez kolejności),
- `price`, `weight` – liczby,
- `bought` – **zmienna docelowa** (to, co model ma przewidywać).

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   size    5 non-null      str    
 1   color   5 non-null      str    
 2   gender  5 non-null      str    
 3   price   5 non-null      float64
 4   weight  5 non-null      int64  
 5   bought  5 non-null      str    
dtypes: float64(1), int64(1), str(4)
memory usage: 437.0 bytes


In [4]:
df.describe()

,price,weight
count,5.000000,5.000000
mean,119.000000,408.000000
std,48.476799,75.299402
min,79.000000,300.000000
25%,89.000000,380.000000
50%,99.000000,410.000000
75%,129.000000,450.000000
max,199.000000,500.000000


## 3. Zmienna docelowa – `LabelEncoder`

`LabelEncoder` zamienia etykiety tekstowe na liczby `0, 1, 2, ...` (w kolejności alfabetycznej). **Służy tylko do zmiennej docelowej `y`**, nie do cech.

In [5]:
df

,size,color,gender,price,weight,bought
0,XL,red,female,199.0,500,yes
1,L,green,male,89.0,450,no
2,M,blue,male,99.0,300,yes
3,L,green,female,129.0,380,no
4,M,red,female,79.0,410,yes


In [7]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y = le.fit_transform(df['bought'])
y

array([1, 0, 1, 0, 1])

In [8]:
# kolejność klas: indeks w tablicy = wartość liczbowa
le.classes_

array(['no', 'yes'], dtype=object)

In [9]:
# powrót do etykiet tekstowych (np. po predykcji modelu)
le.inverse_transform(y)

array(['yes', 'no', 'yes', 'no', 'yes'], dtype=object)

## 4. Cechy porządkowe – `OrdinalEncoder`

Jeśli kategorie mają naturalną kolejność (`M < L < XL`), kodujemy je kolejnymi liczbami za pomocą `OrdinalEncoder`. Domyślnie kolejność jest **alfabetyczna**, czyli zwykle zła:

In [11]:
df

,size,color,gender,price,weight,bought
0,XL,red,female,199.0,500,yes
1,L,green,male,89.0,450,no
2,M,blue,male,99.0,300,yes
3,L,green,female,129.0,380,no
4,M,red,female,79.0,410,yes


In [10]:
from sklearn.preprocessing import OrdinalEncoder

# domyślnie: L=0, M=1, XL=2 - rozmiar M wypada "większy" niż L
OrdinalEncoder().fit_transform(df[['size']])

array([[2.],
       [0.],
       [1.],
       [0.],
       [1.]])

Właściwą kolejność podajemy w parametrze `categories`:

In [12]:
ord_enc = OrdinalEncoder(categories=[['M', 'L', 'XL']])
ord_enc.fit_transform(df[['size']])

array([[2.],
       [1.],
       [0.],
       [1.],
       [0.]])

## 5. Cechy nominalne – `OneHotEncoder`

Dla kolorów nie ma kolejności. Gdybyśmy zakodowali `red=0, green=1, blue=2`, model uznałby, że niebieski jest „większy” od zielonego. Dlatego każda kategoria dostaje **własną kolumnę 0/1** (*one-hot encoding*).

In [13]:
df[['color']]

,color
0,red
1,green
2,blue
3,green
4,red


In [14]:
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(sparse_output=False)   # sparse_output=False: zwykła tablica NumPy
ohe.fit_transform(df[['color']])

array([[0., 0., 1.],
       [0., 1., 0.],
       [1., 0., 0.],
       [0., 1., 0.],
       [0., 0., 1.]])

In [15]:
# nazwy utworzonych kolumn
ohe.get_feature_names_out()

array(['color_blue', 'color_green', 'color_red'], dtype=object)

In [16]:
# wynik jako DataFrame
pd.DataFrame(ohe.transform(df[['color']]), columns=ohe.get_feature_names_out(), index=df.index)

,color_blue,color_green,color_red
0,0.0,0.0,1.0
1,0.0,1.0,0.0
2,1.0,0.0,0.0
3,0.0,1.0,0.0
4,0.0,0.0,1.0


**`drop='first'`** usuwa pierwszą kolumnę. Kolumny one-hot są od siebie zależne (suma w wierszu zawsze wynosi 1), co szkodzi np. regresji liniowej. Brak wartości w pozostałych kolumnach oznacza usuniętą kategorię.

In [13]:
ohe_drop = OneHotEncoder(drop='first', sparse_output=False)
pd.DataFrame(ohe_drop.fit_transform(df[['color']]), columns=ohe_drop.get_feature_names_out(), index=df.index)

,color_green,color_red
0,0.0,1.0
1,1.0,0.0
2,0.0,0.0
3,1.0,0.0
4,0.0,1.0


**`handle_unknown='ignore'`** zabezpiecza przed kategorią, której encoder nie widział podczas `fit` (nowe dane mogą zawierać np. kolor `yellow`). Bez tego parametru pojawi się błąd, z nim nieznana kategoria dostaje same zera.

In [14]:
ohe_safe = OneHotEncoder(handle_unknown='ignore', sparse_output=False).fit(df[['color']])

nowe = pd.DataFrame({'color': ['yellow', 'red']})
ohe_safe.transform(nowe)

array([[0., 0., 0.],
       [0., 0., 1.]])

## 6. `pd.get_dummies()` a `OneHotEncoder`

Pandas ma własną funkcję do kodowania one-hot. Jest wygodna podczas **eksploracji** danych:

In [17]:
pd.get_dummies(df[['size', 'color']], dtype=int)

,size_L,size_M,size_XL,color_blue,color_green,color_red
0,0,0,1,0,0,1
1,1,0,0,0,1,0
2,0,1,0,1,0,0
3,1,0,0,0,1,0
4,0,1,0,0,0,1


In [16]:
# drop_first=True odpowiada drop='first'
pd.get_dummies(df[['size', 'color']], drop_first=True, dtype=int)

,size_M,size_XL,color_green,color_red
0,0,1,0,1
1,0,0,1,0
2,1,0,0,0
3,0,0,1,0
4,1,0,0,1


`get_dummies()` **nie zapamiętuje kategorii**. Dla nowych danych może utworzyć inny zestaw kolumn niż podczas treningu, a model wymaga identycznych kolumn:

In [17]:
pd.get_dummies(df[['color']], dtype=int).columns.tolist()

['color_blue', 'color_green', 'color_red']

In [18]:
pd.get_dummies(nowe, dtype=int).columns.tolist()

['color_red', 'color_yellow']

W projektach ML używamy zatem `OneHotEncoder`, który uczy się kategorii w `fit` i stosuje je w `transform`.

## 7. Skalowanie – `StandardScaler` i `MinMaxScaler`

Cena (79–199) i waga (300–500) mają różne zakresy. Modele oparte na odległościach (KNN, SVM) i na gradiencie (regresja, sieci neuronowe) „widzą” wtedy cechę o większych liczbach jako ważniejszą. Skalowanie wyrównuje cechy.

### StandardScaler (standaryzacja)

$$z = \frac{x - \mu}{\sigma}$$

Po transformacji każda kolumna ma średnią $0$ i odchylenie standardowe $1$.

In [19]:
df[['price', 'weight']]

,price,weight
0,199.0,500
1,89.0,450
2,99.0,300
3,129.0,380
4,79.0,410


In [18]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaled = scaler.fit_transform(df[['price', 'weight']])
scaled

array([[ 1.84506242,  1.3660019 ],
       [-0.69189841,  0.62360956],
       [-0.4612656 , -1.60356745],
       [ 0.2306328 , -0.41573971],
       [-0.92253121,  0.02969569]])

In [20]:
# parametry wyuczone w fit
print('Średnie:', scaler.mean_)
print('Odchylenia:', scaler.scale_)

# kontrola: po standaryzacji średnia ~ 0, odchylenie = 1
print('Średnie po skalowaniu:', scaled.mean(axis=0).round(6))
print('Odchylenia po skalowaniu:', scaled.std(axis=0))

Średnie: [119. 408.]
Odchylenia: [43.35896678 67.34983296]
Średnie po skalowaniu: [0. 0.]
Odchylenia po skalowaniu: [1. 1.]


**Uwaga: dwa odchylenia standardowe.** Pandas liczy odchylenie **próbkowe** (dzielenie przez $n-1$, `ddof=1`), NumPy i scikit-learn – **populacyjne** (dzielenie przez $n$, `ddof=0`). Przy małej próbie różnica jest widoczna, przy dużej – pomijalna.

In [21]:
print('pandas  (ddof=1):', round(df['price'].std(), 2))
print('NumPy   (ddof=0):', round(np.std(df['price'].to_numpy()), 2))
print('sklearn         :', round(scaler.scale_[0], 2))

pandas  (ddof=1): 48.48
NumPy   (ddof=0): 43.36
sklearn         : 43.36


### MinMaxScaler (normalizacja do przedziału 0-1)

$$x' = \frac{x - x_{min}}{x_{max} - x_{min}}$$

Najmniejsza wartość zamienia się w $0$, największa w $1$.

In [22]:
df[['price', 'weight']]

,price,weight
0,199.0,500
1,89.0,450
2,99.0,300
3,129.0,380
4,79.0,410


In [23]:
from sklearn.preprocessing import MinMaxScaler

minmax = MinMaxScaler()
minmax.fit_transform(df[['price', 'weight']])

array([[1.        , 1.        ],
       [0.08333333, 0.75      ],
       [0.16666667, 0.        ],
       [0.41666667, 0.4       ],
       [0.        , 0.55      ]])

In [27]:
data = {
    'size': ['XL', 'L', 'M', 'L', 'M'],
    'color': ['red', 'green', 'blue', 'green', 'red'],
    'gender': ['female', 'male', 'male', 'female', 'female'],
    'price': [199.0, 89.0, 99.0, 129.0, 79.0],
    'weight': [500, 450, 300, 380, 410],
    'bought': ['yes', 'no', 'yes', 'no', 'yes']
}

df_raw = pd.DataFrame(data)
df2 = df_raw.copy()   # pracujemy na kopii, oryginał zostaje do porównań
df2

,size,color,gender,price,weight,bought
0,XL,red,female,199.0,500,yes
1,L,green,male,89.0,450,no
2,M,blue,male,99.0,300,yes
3,L,green,female,129.0,380,no
4,M,red,female,79.0,410,yes


In [29]:
from sklearn.preprocessing import MinMaxScaler

minmax = MinMaxScaler()
minmax.fit(df[['weight']])

,"feature_range feature_range: tuple (min, max), default=(0, 1)Desired range of transformed data.","(0, ...)"
,"copy copy: bool, default=TrueSet to False to perform inplace row normalization and avoid acopy (if the input is already a numpy array).",True
,"clip clip: bool, default=FalseSet to True to clip transformed values of held-out data toprovided `feature_range`.Since this parameter will clip values, `inverse_transform` may notbe able to restore the original data... note:: Setting `clip=True` does not prevent feature drift (a distribution shift between training and test data). The transformed values are clipped to the `feature_range`, which helps avoid unintended behavior in models sensitive to out-of-range inputs (e.g. linear models). Use with care, as clipping can distort the distribution of test data... versionadded:: 0.24",False
Name,Type,Value
"data_max_ data_max_: ndarray of shape (n_features,)Per feature maximum seen in the data.. versionadded:: 0.17 *data_max_*","ndarray[float64](1,)",[500.]
"data_min_ data_min_: ndarray of shape (n_features,)Per feature minimum seen in the data.. versionadded:: 0.17 *data_min_*","ndarray[float64](1,)",[300.]
"data_range_ data_range_: ndarray of shape (n_features,)Per feature range ``(data_max_ - data_min_)`` seen in the data.. versionadded:: 0.17 *data_range_*","ndarray[float64](1,)",[200.]
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](1,)",['weight']
"min_ min_: ndarray of shape (n_features,)Per feature adjustment for minimum. Equivalent to``min - X.min(axis=0) * self.scale_``","ndarray[float64](1,)",[-1.5]
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,1
"n_samples_seen_ n_samples_seen_: intThe number of samples processed by the estimator.It will be reset on new calls to fit, but increments across``partial_fit`` calls.",int,5


In [40]:
minmax.transform([[300]])

/Users/tomasz/Desktop/ML_DL_szkolenie_23_11_26/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


array([[0.]])

**Którego skalera użyć?**
- `StandardScaler` – domyślny wybór (regresja liniowa i logistyczna, SVM, KNN, sieci neuronowe).
- `MinMaxScaler` – gdy potrzebny jest sztywny przedział $[0, 1]$ (np. piksele obrazów). Jest wrażliwy na wartości odstające.
- Drzewa decyzyjne i lasy losowe **nie wymagają** skalowania.

## 8. Zasada: `fit` tylko na zbiorze treningowym

Skaler uczy się średniej i odchylenia. Jeśli policzymy je na **całym** zbiorze, informacja ze zbioru testowego „przecieka” do modelu (*data leakage*), a wynik oceny jest zawyżony.

**Reguła:** `fit` (lub `fit_transform`) na zbiorze treningowym, a na testowym tylko `transform`. Podział danych na zbiór treningowy i testowy omawialiśmy w notatniku 2.3/06, a w praktyce zobaczymy go w rozdziale o regresji liniowej.

In [24]:
train = df[['price']].iloc[:4]    # 4 pierwsze wiersze
test = df[['price']].iloc[4:]     # ostatni wiersz

scaler = StandardScaler().fit(train)
print('Średnia i odchylenie z danych treningowych:', scaler.mean_, scaler.scale_)

scaler.transform(test)

Średnia i odchylenie z danych treningowych: [129.] [43.01162634]


array([[-1.16247639]])

In [25]:
train

,price
0,199.0
1,89.0
2,99.0
3,129.0


In [26]:
test

,price
4,79.0


## 9. Wszystko razem – `ColumnTransformer`

Każda grupa kolumn wymaga innej operacji. `ColumnTransformer` pozwala przypisać transformery do kolumn i wykonać wszystko jedną komendą. Zapamiętuje parametry z `fit`, więc te same kroki można później zastosować do nowych danych.

In [41]:
df

,size,color,gender,price,weight,bought
0,XL,red,female,199.0,500,yes
1,L,green,male,89.0,450,no
2,M,blue,male,99.0,300,yes
3,L,green,female,129.0,380,no
4,M,red,female,79.0,410,yes


In [45]:
y

array([1, 0, 1, 0, 1])

In [43]:
from sklearn.compose import ColumnTransformer

X = df_raw.drop(columns='bought')
y = LabelEncoder().fit_transform(df_raw['bought'])

preprocessor = ColumnTransformer(
    transformers=[
        ('rozmiar', OrdinalEncoder(categories=[['M', 'L', 'XL']]), ['size']),
        ('kategorie', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ['color', 'gender']),
        ('liczby', StandardScaler(), ['price', 'weight']),
    ],
    verbose_feature_names_out=False,
)
preprocessor.set_output(transform='pandas')   # wynik jako DataFrame

X_ready = preprocessor.fit_transform(X)
X_ready

,size,color_blue,color_green,color_red,gender_female,gender_male,price,weight
0,2.0,0.0,0.0,1.0,1.0,0.0,1.845062,1.366002
1,1.0,0.0,1.0,0.0,0.0,1.0,-0.691898,0.623610
2,0.0,1.0,0.0,0.0,0.0,1.0,-0.461266,-1.603567
3,1.0,0.0,1.0,0.0,1.0,0.0,0.230633,-0.415740
4,0.0,0.0,0.0,1.0,1.0,0.0,-0.922531,0.029696


Dane są gotowe dla modelu: same liczby, wyrównane skale. Te same kroki zastosujemy do nowego wiersza (nawet z nieznanym kolorem):

In [25]:
nowy = pd.DataFrame({'size': ['L'], 'color': ['yellow'], 'gender': ['male'],
                     'price': [149.0], 'weight': [420]})
preprocessor.transform(nowy)

,size,color_blue,color_green,color_red,gender_female,gender_male,price,weight
0,1.0,0.0,0.0,0.0,0.0,1.0,0.691898,0.178174


W kolejnych rozdziałach `ColumnTransformer` połączymy z modelem w jeden obiekt (`Pipeline`).

## Zapis do pliku


In [46]:
import pickle

with open('preprocessor.pkl', 'wb') as f:
    pickle.dump(preprocessor, f)

In [47]:
with open('preprocessor.pkl', 'rb') as f:
    preprocessor = pickle.load(f)

## 10. Podsumowanie

| Zadanie | Narzędzie | Uwagi |
|---|---|---|
| Kodowanie zmiennej docelowej | `LabelEncoder` | tylko `y` |
| Kategorie z kolejnością | `OrdinalEncoder(categories=...)` | podaj kolejność ręcznie |
| Kategorie bez kolejności | `OneHotEncoder` | `drop='first'` dla modeli liniowych, `handle_unknown='ignore'` dla nowych danych |
| Skalowanie | `StandardScaler`, `MinMaxScaler` | niepotrzebne dla drzew |
| Łączenie kroków | `ColumnTransformer` | jedna transformacja dla wszystkich kolumn |

**Najważniejsze zasady:**
1. `fit` tylko na danych treningowych, `transform` na treningowych i testowych.
2. Kolejność kategorii porządkowych podajemy jawnie.
3. W projektach ML preferujemy `OneHotEncoder` zamiast `pd.get_dummies()`.